> ## SOLUTION / ANSWER KEY &mdash; Lab 9.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-assistive-not-autonomous.ipynb`](../lab-10-assistive-not-autonomous.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 9.10 &mdash; Assistive, Not Autonomous

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Return analysis flagged for review -- never a decision
- Require every claim to be cited so review is genuine
- Keep a human as the owner of any consequential decision

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; grounding/citation/compute logic and, in the agent-assembly labs, tool wiring &amp; the read-only guardrail &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It grounds &amp; cites every figure, gives **no advice**, and has **no trade tool** &mdash; a human analyst decides.

**Reference:** [Module 9 slides &mdash; Assistive on judgement, autonomous on legwork](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-09-10"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
The golden rule of Day 5 (deck slides 11, 17): agents are **assistive, not autonomous**. The insight
agent does the grounded legwork and returns analysis flagged **`needs_review`** &mdash; it never
decides. And to defend against **automation bias** (humans rubber-stamping a confident machine), the
review is only real if the agent **shows its work**: every claim **cited**. The **human** owns any
consequential decision.

In [ ]:
# The agent's output is analysis + a needs_review flag -- never "executed" or a recommendation.
print("assistive output shape:", {"summary": "...", "status": "needs_review", "claims": ["..."]})

## Your Turn
Complete `make_insight` (flag for review), `reviewable` (require citations), and `owns_decision`.

In [ ]:
def make_insight(summary, claims):
    # analysis + a needs_review flag -- the agent never decides
    return {"summary": summary, "status": "needs_review", "claims": claims}

def reviewable(insight):
    # a review is only genuine if EVERY claim is cited (defends against automation bias)
    return all(c.get("source") for c in insight["claims"])

def owns_decision(insight):
    # who owns any consequential decision -- never the agent
    return "human"

try:
    claims = [{'metric': 'revenue', 'source': 'p4'}, {'metric': 'margin', 'source': 'p4'}]
    ins = make_insight('Revenue +12% YoY [p4]; margin down [p4].', claims)
    print('status    :', ins['status'])
    print('reviewable:', reviewable(ins))
    print('decides   :', owns_decision(ins))
    uncited = make_insight('...', [{'metric': 'guess', 'source': ''}])
    print('uncited reviewable?', reviewable(uncited))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the output is flagged needs_review", lambda: make_insight("s", [])["status"] == "needs_review")
expect_true("the agent never marks a decision as executed", lambda: make_insight("s", [])["status"] != "executed")
expect_true("a fully-cited insight is genuinely reviewable", lambda: reviewable(make_insight("s", [{"metric": "revenue", "source": "p4"}])) is True)
expect_true("an uncited claim makes review a rubber-stamp (not reviewable)", lambda: reviewable(make_insight("s", [{"metric": "guess", "source": ""}])) is False)
expect_true("a human owns the decision", lambda: owns_decision(make_insight("s", [])) == "human")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; the real LangChain interface (not graded)
See how the real LangChain exposes structured, citable output (the shape a human reviews). It needs only `pip install langchain-core` (already in the course env) and makes **no** network
call. **The graded steps above never call an LLM, so the lab always verifies offline.**

In [ ]:
try:
    from langchain_core.output_parsers import JsonOutputParser
    print("Real LangChain: bind a schema (e.g. Pydantic) so the model MUST return {summary, claims[], each cited}.")
    print("JsonOutputParser / with_structured_output enforce the shape a human reviews -- citations included.")
    print("Attach a Langfuse/LangSmith callback and the whole assistive run is logged for audit.")
except Exception as e:
    print("Install langchain-core to explore structured output -- skipping:", type(e).__name__)
print("The assistive, needs_review, fully-cited output above already runs offline.")

---
### Key takeaway
Assistive, not autonomous: the agent flags analysis for review and never decides, and citations make the review genuine rather than a rubber-stamp. The human owns every consequential decision.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>